In [1]:
pip install -q boto3

Note: you may need to restart the kernel to use updated packages.


In [3]:

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split


# ============================================================
# 1. LOAD IRIS DATASET
# ============================================================

print("=" * 70)
print("[1] Loading Iris dataset")
print("=" * 70)

iris = load_iris()

X = iris.data.astype(np.float32)
y = iris.target.astype(np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", iris.target_names)


# ============================================================
# 2. TRAIN / TEST SPLIT
# ============================================================

print("\n" + "=" * 70)
print("[2] Splitting dataset")
print("=" * 70)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


# ============================================================
# 3. NORMALIZATION
# ============================================================

print("\n" + "=" * 70)
print("[3] Creating normalization layer")
print("=" * 70)

normalizer = tf.keras.layers.Normalization()

normalizer.adapt(X_train)

print("Normalization completed")


# ============================================================
# 4. BUILD MODEL
# ============================================================

print("\n" + "=" * 70)
print("[4] Building TensorFlow model")
print("=" * 70)

model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(4,),
        name="iris_features"
    ),

    normalizer,

    tf.keras.layers.Dense(
        16,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        8,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        3,
        activation="softmax",
        name="prediction"
    )
])

model.summary()


# ============================================================
# 5. COMPILE MODEL
# ============================================================

print("\n" + "=" * 70)
print("[5] Compiling model")
print("=" * 70)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully")


# ============================================================
# 6. TRAIN MODEL
# ============================================================

print("\n" + "=" * 70)
print("[6] Training model")
print("=" * 70)

history = model.fit(
    X_train,
    y_train,
    validation_data=(
        X_test,
        y_test
    ),
    epochs=30,
    batch_size=16,
    verbose=1
)


# ============================================================
# 7. EVALUATE MODEL
# ============================================================

print("\n" + "=" * 70)
print("[7] Evaluating model")
print("=" * 70)

loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test loss:", loss)
print("Test accuracy:", accuracy)


# ============================================================
# 8. CREATE LOCAL MODEL DIRECTORY
# ============================================================

print("\n" + "=" * 70)
print("[8] Creating model directory")
print("=" * 70)

MODEL_DIR = "/home/jovyan/Iris_KubeFlow/iris_model/1"

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

print("Model directory:")
print(MODEL_DIR)


# ============================================================
# 9. EXPORT AS TENSORFLOW SAVEDMODEL
# ============================================================

print("\n" + "=" * 70)
print("[9] Exporting TensorFlow SavedModel")
print("=" * 70)

# TensorFlow 2.16+ / Keras 3:
# model.export() creates TensorFlow SavedModel.
#
# Do NOT use:
# model.save(MODEL_DIR)

model.export(
    MODEL_DIR
)

print("\nSavedModel export completed!")


# ============================================================
# 10. VERIFY SAVEDMODEL
# ============================================================

print("\n" + "=" * 70)
print("[10] Verifying saved model")
print("=" * 70)

saved_model_pb = os.path.join(
    MODEL_DIR,
    "saved_model.pb"
)

variables_dir = os.path.join(
    MODEL_DIR,
    "variables"
)

print(
    "saved_model.pb exists:",
    os.path.isfile(saved_model_pb)
)

print(
    "variables directory exists:",
    os.path.isdir(variables_dir)
)


# ============================================================
# 11. SHOW MODEL FILES
# ============================================================

print("\n" + "=" * 70)
print("[11] Saved model files")
print("=" * 70)

for root, dirs, files in os.walk(
    MODEL_DIR
):
    for file in files:

        file_path = os.path.join(
            root,
            file
        )

        file_size = os.path.getsize(
            file_path
        )

        print(
            file_path,
            "->",
            file_size,
            "bytes"
        )


# ============================================================
# 12. FINAL VALIDATION
# ============================================================

if not os.path.isfile(
    saved_model_pb
):
    raise RuntimeError(
        "saved_model.pb was NOT created!"
    )

if not os.path.isdir(
    variables_dir
):
    raise RuntimeError(
        "variables directory was NOT created!"
    )


print("\n" + "=" * 70)
print("MODEL SAVED SUCCESSFULLY")
print("=" * 70)

print("\nLocal SavedModel path:")

print(
    MODEL_DIR
)

print("\nTest accuracy:")

print(
    accuracy
)


2026-08-09 05:22:18.908353: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-09 05:22:18.919480: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-09 05:22:19.007417: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-09 05:22:19.117140: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-09 05:22:19.207352: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

[1] Loading Iris dataset
X shape: (150, 4)
y shape: (150,)
Classes: ['setosa' 'versicolor' 'virginica']

[2] Splitting dataset
Training samples: 120
Testing samples: 30

[3] Creating normalization layer
Normalization completed

[4] Building TensorFlow model


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization (Normalization)   │ (None, 4)              │             9 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ prediction (Dense)              │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 252 (1012.00 B)

 Trainable params: 243 (972.00 B)

 Non-trainable params: 9 (40.00 B)


[5] Compiling model
Model compiled successfully

[6] Training model
Epoch 1/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.2372 - loss: 1.3310 - val_accuracy: 0.3000 - val_loss: 1.2396
Epoch 2/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2799 - loss: 1.2322 - val_accuracy: 0.3667 - val_loss: 1.1915
Epoch 3/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2835 - loss: 1.1777 - val_accuracy: 0.3667 - val_loss: 1.1452
Epoch 4/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2599 - loss: 1.1372 - val_accuracy: 0.5000 - val_loss: 1.1024
Epoch 5/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4304 - loss: 1.0969 - val_accuracy: 0.6667 - val_loss: 1.0638
Epoch 6/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5254 - loss: 1.0650 - val_accuracy: 0.6667 - val_loss: 1.0294
Epoch 7/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6833 - loss: 1.0113 - val_accuracy: 0.7000 - val_loss: 0.9975
Epoch 8/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accur

INFO:tensorflow:Assets written to: /home/jovyan/Iris_KubeFlow/iris_model/1/assets


Saved artifact at '/home/jovyan/Iris_KubeFlow/iris_model/1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 4), dtype=tf.float32, name='iris_features')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  123730880676304: TensorSpec(shape=(1, 4), dtype=tf.float32, name=None)
  123730880675920: TensorSpec(shape=(1, 4), dtype=tf.float32, name=None)
  123730885391952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123730885394256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123730885394064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123730880677264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123730880677648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123730880678416: TensorSpec(shape=(), dtype=tf.resource, name=None)

SavedModel export completed!

[10] Verifying saved model
saved_model.pb exists: True
variables directory exists: True

[11] Saved model